# DMI visualization demo

Reads activations from ClickHouse (written by `run_offload_hf.py` or `run_offload_vllm.py`) and renders four mech-interp views of the IOI prompt run on Qwen3-0.6B:

1. **Attention patterns** -- per (layer, head) heatmap of attention weights.  HF only (vLLM excludes attention weights).
2. **Residual-stream norm by layer** -- L2 norm per (layer, token), one trace per token.
3. **Per-token confidence** -- each generated token shaded by the model's top-1 probability.
4. **Top-k alternative tokens** -- each generated token's log-prob with the model's top-k shortlist on hover.

Switch `MODEL_ID` in the setup cell between `"demo_hf"` and `"demo_vllm"` to render the corresponding offload run.  The attention-pattern cell (plot 1) is a no-op when reading the vLLM slot.

In [ ]:
# Setup: connect to CH, load tokenizer, decode token_ids for labels.
import os

import clickhouse_driver
import circuitsvis as cv
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display
from transformers import AutoTokenizer

# Inject CSS so CircuitsVis widgets render at a comfortable size and
# their hover tooltips don't get clipped by the output cell boundary.
# CircuitsVis uses Popper.js (popper / tooltip class names + data-popper-*
# attributes) for hover popups; we lift overflow restrictions on the
# Jupyter output containers and bump the popup's z-index above
# everything else.  The min-height on the CV container ensures the
# tooltip has room to render below the token without flipping.
display(HTML("""
<style>
  /* Allow tooltips to escape every Jupyter output container */
  .jp-Cell-outputArea, .jp-OutputArea, .jp-OutputArea-child,
  .jp-OutputArea-output, .jp-OutputArea-output-data,
  .output_area, .output_subarea, .output_html {
    overflow: visible !important;
  }
  /* Bigger CircuitsVis container with enough vertical room for the popup */
  div[id^="circuits-vis-"] {
    min-width: 100%;
    min-height: 220px;
    overflow: visible !important;
    font-size: 16px;
    line-height: 1.8;
  }
  div[id^="circuits-vis-"] * { overflow: visible !important; }
  /* Float the tooltip / popper above all other UI */
  .tooltip, .popper, [data-popper-placement],
  div[id^="circuits-vis-"] [class*="ooltip"],
  div[id^="circuits-vis-"] [class*="opper"] {
    z-index: 99999 !important;
    position: absolute !important;
  }
</style>
"""))

# Two well-known slots written by run_offload_hf.py / run_offload_vllm.py.
# Switch to "demo_vllm" to render the vLLM run.
MODEL_ID = "demo_hf"
HF_MODEL = "Qwen/Qwen3-0.6B"
DB_HOST = os.environ.get("DMX_DB_HOST", "localhost")
DB_PORT = int(os.environ.get("DMX_DB_PORT", "9000"))

# ---------------------------------------------------------------------------
# Vendored CH-blob -> NumPy decoder helpers.
# Mirrors tests/compare_disk_vs_ch.py (kept self-contained for the demo).
# ---------------------------------------------------------------------------

_DTYPE_MAP = {
    "torch.bfloat16": torch.bfloat16, "torch.float": torch.float32,
    "torch.half": torch.float16, "torch.float16": torch.float16,
    "torch.int": torch.int32, "torch.long": torch.int64,
    "torch.uint8": torch.uint8, "torch.int8": torch.int8,
    "torch.short": torch.int16, "torch.double": torch.float64,
    "torch.bool": torch.bool,
}

_BUF_TO_CH_ACT = {
    "resid_pre":    "blocks.hook_resid_pre",
    "pattern":      "blocks.attn.hook_pattern",
    "mlp_post":     "blocks.hook_mlp_post",
    "final_logits": "final_logits",
    "token_ids":    "token_ids",
}


def _decode_str(v):
    return v.decode() if isinstance(v, bytes) else v


def _query_rows(client, model_id, act_name):
    rows = client.execute(
        "SELECT request_id, layer_no, shard_rank, start_token_idx, end_token_idx, "
        "dtype, shape, bytes FROM default.offload "
        "WHERE model_id = %(m)s AND act_name = %(a)s "
        "ORDER BY layer_no, start_token_idx",
        {"m": model_id, "a": act_name},
        settings={"strings_as_bytes": True},
    )
    decoded = []
    for req_id, layer_no, shard, s, e, dtype_str, shape, payload in rows:
        dt = _DTYPE_MAP.get(_decode_str(dtype_str), torch.float32)
        t = torch.frombuffer(bytearray(payload), dtype=dt).reshape(list(shape))
        decoded.append({
            "req_id": _decode_str(req_id),
            "layer": int(layer_no), "shard": int(shard),
            "start": int(s), "end": int(e),
            "tensor": t,
        })
    return decoded


def fetch_token_ids(client, model_id):
    """Return concatenated list[int] of token IDs (prompt + generated)."""
    rows = _query_rows(client, model_id, _BUF_TO_CH_ACT["token_ids"])
    if not rows:
        raise RuntimeError(f"No token_ids rows for model_id={model_id!r}.  "
                           "Run the offload script first.")
    rows.sort(key=lambda r: r["start"])
    return torch.cat([r["tensor"].long().flatten() for r in rows]).tolist()


def fetch_per_layer(client, model_id, hook_short):
    """Return torch.Tensor [num_layers, total_tokens, ...] for a per-layer
    hook whose chunks share all dims except dim 0 (the token dim).
    Suitable for resid_pre, mlp_post (NOT for pattern -- use
    fetch_attention_pattern instead).  Returns None if no rows."""
    rows = _query_rows(client, model_id, _BUF_TO_CH_ACT[hook_short])
    if not rows:
        return None
    by_layer = {}
    for r in rows:
        by_layer.setdefault(r["layer"], []).append(r)
    num_layers = max(by_layer) + 1
    layer_tensors = []
    for L in range(num_layers):
        chunks = sorted(by_layer.get(L, []), key=lambda r: r["start"])
        if not chunks:
            return None
        layer_tensors.append(torch.cat([c["tensor"].float() for c in chunks], dim=0))
    return torch.stack(layer_tensors, dim=0)


def fetch_global(client, model_id, hook_short):
    """Return torch.Tensor [total_tokens, ...] for a non-per-layer hook."""
    rows = _query_rows(client, model_id, _BUF_TO_CH_ACT[hook_short])
    if not rows:
        return None
    rows.sort(key=lambda r: r["start"])
    return torch.cat([r["tensor"].float() for r in rows], dim=0)


def fetch_attention_pattern(client, model_id):
    """Reconstruct full [num_layers, num_heads, total_seq, total_seq]
    attention pattern from per-step chunks.

    Each chunk has shape [H, q_chunk, kv_chunk] where kv_chunk grows as
    decoding progresses (causal mask): the prefill chunk is [H, P, P],
    each decode step adds [H, 1, P+k] where k is the decode step.  We
    zero-fill an [L, H, total_seq, total_seq] tensor and place each chunk
    into its (q_range, kv_range) slot.  Upper-triangle cells stay zero
    (model never attends to future positions)."""
    rows = _query_rows(client, model_id, _BUF_TO_CH_ACT["pattern"])
    if not rows:
        return None
    by_layer = {}
    for r in rows:
        by_layer.setdefault(r["layer"], []).append(r)
    num_layers = max(by_layer) + 1
    n_heads = rows[0]["tensor"].shape[0]
    total_seq = max(r["end"] for r in rows)
    pattern = torch.zeros(num_layers, n_heads, total_seq, total_seq, dtype=torch.float32)
    for L_idx, chunks in by_layer.items():
        for c in chunks:
            t = c["tensor"].float()    # [H, q_chunk, kv_chunk]
            s, e = c["start"], c["end"]
            kv_len = t.shape[-1]
            pattern[L_idx, :, s:e, :kv_len] = t
    return pattern


# ---------------------------------------------------------------------------
# Connect + load common state.
# ---------------------------------------------------------------------------

client = clickhouse_driver.Client(host=DB_HOST, port=DB_PORT)
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)

token_ids = fetch_token_ids(client, MODEL_ID)
str_tokens = [tokenizer.decode([t]) for t in token_ids]
print(f"MODEL_ID = {MODEL_ID}")
print(f"tokens   = {str_tokens}")

## 1. Attention patterns

What to look for: in the IOI prompt (`When John and Mary went to the store, John gave Mary a`), the model has to figure out that the indirect object (`Mary`) should NOT be the next-word target.  Late-layer name-mover heads track which name has been mentioned and shape the residual stream accordingly.  Look for heads in layers 12-25 with a clear column over `Mary` or `John` -- those are the canonical IOI circuit components.

Heads in earlier layers usually show simpler patterns (positional, BOS-attending, previous-token).

Skipped automatically when reading the vLLM slot (`hook_pattern` isn't captured under vLLM).

In [ ]:
from IPython.display import display

pattern = fetch_attention_pattern(client, MODEL_ID)
if pattern is None:
    print(f"No 'pattern' rows for {MODEL_ID!r} (expected when reading the vLLM slot).")
else:
    # pattern shape: [num_layers, num_heads, total_seq, total_seq] (zero-filled
    # upper triangle).  Pick a layer to display; CircuitsVis adds a head
    # dropdown.  Layer 0 is positional; late layers (>20) hold the IOI circuit
    # heads -- flip LAYER to explore.
    LAYER = 0
    L, H, T, _ = pattern.shape
    print(f"attention pattern: layers={L}  heads={H}  total_seq={T}  rendering layer {LAYER}")
    display(cv.attention.attention_patterns(
        tokens=str_tokens[:T],
        attention=pattern[LAYER],   # [H, T, T]
    ))

## 2. Residual-stream norm by layer

Diagnostic plot: for each token position, the L2 norm of the residual stream at each layer.  In a healthy model, norm grows smoothly with depth.  Spikes on individual tokens are "outlier features" -- a known phenomenon (Dettmers et al's int8 quantization paper flagged these as the reason naive int8 breaks).  Flat layers suggest dead contributions.  Sudden jumps suggest training artifacts.

In [ ]:
resid = fetch_per_layer(client, MODEL_ID, "resid_pre")
if resid is None:
    print(f"No 'resid_pre' rows for {MODEL_ID!r}.")
else:
    # resid shape: [num_layers, total_tokens, hidden].
    norms = resid.norm(dim=-1).numpy()  # [num_layers, total_tokens]
    L, T = norms.shape
    print(f"resid_pre: layers={L}  tokens={T}")
    plt.figure(figsize=(11, 4.5))
    for tok_idx in range(min(T, 12)):
        plt.plot(norms[:, tok_idx], label=f"{tok_idx}: {str_tokens[tok_idx]!r}")
    plt.xlabel("layer"); plt.ylabel(r"$\|resid_{pre}\|_2$")
    plt.title(f"Residual-stream norm by layer ({MODEL_ID})")
    plt.legend(loc="upper left", fontsize=8)
    plt.tight_layout(); plt.show()

## 3. Per-token confidence

Each generated token shaded by the model's top-1 probability when it picked that token.  Bright cells = the model was confident; dim cells = the model was unsure (multiple plausible alternatives).  In greedy decoding, the chosen token is always the argmax, so this also reads as "how peaked was the distribution at this step?".

In [ ]:
logits = fetch_global(client, MODEL_ID, "final_logits")
if logits is None:
    print(f"No 'final_logits' rows for {MODEL_ID!r}.")
else:
    # logits shape: [N, vocab].  N forward passes, but token_ids only
    # captures forward inputs (= prompt + first N-1 generated tokens),
    # so we can label N - 1 tokens with their probabilities.
    probs = torch.softmax(logits, dim=-1)
    top1_prob = probs.max(dim=-1).values.numpy()
    N = top1_prob.shape[0]
    n_show = N - 1
    gen_str_tokens = str_tokens[-n_show:]
    top1_show = top1_prob[:n_show]
    print(f"final_logits: {N} rows; rendering {n_show} captured generated tokens "
          f"({gen_str_tokens})")
    display(cv.tokens.colored_tokens(
        tokens=gen_str_tokens,
        values=top1_show.tolist(),
    ))

## 4. Top-k alternative tokens per position

For each generated token, the model considered a *distribution* of next tokens.  This view colors each token by its log-probability under the model and shows the top-10 alternatives on hover, plus the rank of the token actually chosen.  After `Mary a`, plausible alternatives in IOI runs include `gift`, `book`, `kiss`, `letter`, `present`.  Looking at the shortlist often tells you more than the single chosen token does.

In [ ]:
if logits is None:
    print(f"No 'final_logits' rows for {MODEL_ID!r}.")
else:
    log_probs = torch.log_softmax(logits, dim=-1)
    N = log_probs.shape[0]
    seed_tokens = token_ids[-N:]   # N ids: last prompt + N-1 captured gen
    display(cv.logits.token_log_probs(
        token_indices=torch.tensor(seed_tokens),
        log_probs=log_probs,
        to_string=lambda i: tokenizer.decode([int(i)]),
        top_k=10,
    ))